In [2]:
import sys
print(sys.executable)


c:\Users\thion\.conda\envs\learn-env\python.exe


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 


In [4]:
# Clear all dataframes and variables
%reset -f

# Re-import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import kagglehub

print("All variables cleared successfully")

c:\Users\thion\.conda\envs\learn-env\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (1.26.20) or chardet (6.0.0.post1)/charset_normalizer (2.1.1) doesn't match a supported version!
  warnings.warn(


All variables cleared successfully


In [5]:
# Data Collection for Kenya Monetary Policy Project
# Goal: Load and explore macroeconomic and financial data

In [3]:
import pandas as pd
import os
import kagglehub

# 1. Download/Get path
path = kagglehub.dataset_download("macmini62/nairobi-securities-exchangense-stocks-data")
all_files = [os.path.join(path, f) for f in os.listdir(path) if f.endswith('.csv')]

# 2. Loop and Combine
li = []
for filename in all_files:
    df_year = pd.read_csv(filename)
    
    # Standardize column names to handle 'CODE' vs 'code' vs 'Ticker'
    df_year.columns = [c.upper().strip() for c in df_year.columns]
    
    # Rename 'TICKER' to 'CODE' if that's what the newer files use
    if 'TICKER' in df_year.columns:
        df_year = df_year.rename(columns={'TICKER': 'CODE'})
        
    li.append(df_year)

nasi_cbr = pd.concat(li, axis=0, ignore_index=True)

# 3. Filter for the Index
# Note: In NSE data, the All Share Index is usually coded as 'NASI' or 'NSEASI'
# The 20-Share Index is 'NSE20'
nasi_data = nasi_cbr[nasi_cbr['CODE'].str.contains('NASI', case=False, na=False)].copy()

# 4. Sort by Date
nasi_data['DATE'] = pd.to_datetime(nasi_data['DATE'], format='mixed', dayfirst=True)
nasi_data = nasi_data.sort_values(by='DATE')

# Save the combined file
nasi_data.to_csv(r"C:\Users\thion\Kenya-Monetary-Policy-Stock-Market\notebooks\data\raw\nasi_2008_2025.csv", index=False)

In [2]:
print(f"Merged {len(all_files)} files. Total NASI rows: {len(nasi_data)}")


Merged 19 files. Total NASI rows: 4211


In [4]:
nasi_data.head()

,DATE,CODE,NAME,12M LOW,12M HIGH,DAY LOW,DAY HIGH,DAY PRICE,PREVIOUS,CHANGE,CHANGE%,VOLUME,ADJUST,ADJUSTED,ADJUSTED PRICE
13213,2008-01-04,^NASI,NSE All-Share Index,74.37,122.14,94.71,94.71,94.71,94.64,0.07,0.07%,"5,040,000",NaN,-,NaN
15948,2008-01-07,^NASI,NSE All-Share Index,74.68,122.14,111.71,111.71,111.71,112.11,-0.4,0.36%,"45,880,000",NaN,-,NaN
16985,2008-01-08,^NASI,NSE All-Share Index,73.71,121.58,101.14,101.14,101.14,101.74,-0.6,0.59%,"27,100,000",NaN,-,NaN
17865,2008-01-09,^NASI,NSE All-Share Index,73.71,121.58,97.17,97.17,97.17,97.55,-0.38,0.39%,"12,250,000",NaN,-,NaN
20386,2008-01-12,^NASI,NSE All-Share Index,73.71,121.58,70.32,70.32,70.32,71.28,-0.96,1.37%,"4,300,000",NaN,-,NaN


In [5]:
import io

# ── 1. CBR RATE ──────────────────────────────────────────────────────────────
cbr_raw = """Date,CBR_Rate
2007-01-01,8.50
2008-06-09,9.00
2008-12-01,8.50
2009-01-01,8.25
2009-03-01,8.00
2010-01-01,7.00
2010-03-23,6.75
2010-07-27,6.00
2011-03-22,6.25
2011-05-31,7.00
2011-09-14,7.00
2011-10-05,11.00
2011-11-01,16.50
2011-12-05,18.00
2012-07-05,16.50
2012-09-05,13.00
2012-11-07,11.00
2013-01-10,9.50
2013-05-07,8.50
2015-06-09,10.00
2015-07-07,11.50
2016-05-23,10.50
2016-09-20,10.00
2018-03-19,9.50
2018-07-23,9.00
2019-11-25,8.50
2020-01-27,8.25
2020-03-23,7.25
2020-04-29,7.00
2022-05-30,7.50
2022-09-29,8.25
2022-11-23,8.75
2023-03-29,9.50
2023-06-26,10.50
2023-12-05,12.50
2024-02-06,13.00
2024-08-06,12.75
2024-10-08,12.00
2024-12-04,11.25
2025-02-05,10.75
2026-02-10,8.75"""

cbr_df = pd.read_csv(io.StringIO(cbr_raw))
cbr_df['Date'] = pd.to_datetime(cbr_df['Date'])
cbr_df = cbr_df.sort_values('Date').reset_index(drop=True)

# ── 2. 91-DAY T-BILL RATE (Kenya, monthly) ───────────────────────────────────
# Source: Central Bank of Kenya
tbill_raw = """Date,TBill_91D
2008-01-01,7.18
2008-06-01,7.80
2009-01-01,7.07
2009-06-01,6.37
2010-01-01,3.84
2010-06-01,3.42
2011-01-01,4.60
2011-06-01,9.17
2011-12-01,18.99
2012-06-01,12.99
2012-12-01,9.50
2013-06-01,8.64
2013-12-01,10.24
2014-06-01,9.98
2014-12-01,10.74
2015-06-01,11.92
2015-12-01,11.02
2016-06-01,9.02
2016-12-01,8.08
2017-06-01,8.34
2017-12-01,7.99
2018-06-01,7.70
2018-12-01,7.24
2019-06-01,6.83
2019-12-01,7.22
2020-06-01,6.68
2020-12-01,6.59
2021-06-01,7.00
2021-12-01,7.24
2022-06-01,8.03
2022-12-01,9.37
2023-06-01,12.80
2023-12-01,15.97
2024-06-01,16.07
2024-12-01,12.46
2025-06-01,10.20"""

tbill_df = pd.read_csv(io.StringIO(tbill_raw))
tbill_df['Date'] = pd.to_datetime(tbill_df['Date'])
tbill_df = tbill_df.sort_values('Date').reset_index(drop=True)

# ── 3. KENYA INFLATION RATE (monthly, year-on-year CPI %) ────────────────────
# Source: Kenya National Bureau of Statistics (KNBS)
inflation_raw = """Date,Inflation_Rate
2008-01-01,12.04
2008-06-01,26.60
2008-12-01,17.83
2009-06-01,9.49
2009-12-01,5.32
2010-06-01,3.46
2010-12-01,4.51
2011-06-01,14.49
2011-12-01,18.93
2012-06-01,10.06
2012-12-01,3.20
2013-06-01,4.91
2013-12-01,7.15
2014-06-01,7.44
2014-12-01,6.02
2015-06-01,7.03
2015-12-01,8.01
2016-06-01,5.79
2016-12-01,6.35
2017-06-01,9.21
2017-12-01,4.48
2018-06-01,4.28
2018-12-01,5.72
2019-06-01,5.70
2019-12-01,5.82
2020-06-01,4.36
2020-12-01,5.62
2021-06-01,6.32
2021-12-01,5.74
2022-06-01,7.91
2022-12-01,9.10
2023-06-01,7.90
2023-12-01,6.60
2024-06-01,4.60
2024-12-01,3.00
2025-06-01,3.50"""

inflation_df = pd.read_csv(io.StringIO(inflation_raw))
inflation_df['Date'] = pd.to_datetime(inflation_df['Date'])
inflation_df = inflation_df.sort_values('Date').reset_index(drop=True)

# ── 4. LOAD & PREPARE NASI DATA ───────────────────────────────────────────────
nasi_path = r"C:\Users\thion\Kenya-Monetary-Policy-Stock-Market\notebooks\data\raw\nasi_2008_2025.csv"
nasi_df = pd.read_csv(nasi_path)
nasi_df.columns = [c.capitalize().strip() for c in nasi_df.columns]
nasi_df['Date'] = pd.to_datetime(nasi_df['Date'], dayfirst=True, format='mixed', errors='coerce')
nasi_df = nasi_df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

# ── 5. MERGE ALL THREE VARIABLES ──────────────────────────────────────────────
# Each merge_asof carries the last known value forward to each trading day
final_nasi = pd.merge_asof(nasi_df,   cbr_df,       on='Date', direction='backward')
final_nasi = pd.merge_asof(final_nasi, tbill_df,     on='Date', direction='backward')
final_nasi = pd.merge_asof(final_nasi, inflation_df, on='Date', direction='backward')

# ── 6. SAVE ───────────────────────────────────────────────────────────────────
output_path = r"C:\Users\thion\Kenya-Monetary-Policy-Stock-Market\notebooks\data\processed\nasi_with_macro_final.csv"
final_nasi.to_csv(output_path, index=False)

print(f"✅ Done! Rows: {len(final_nasi)}, Columns: {list(final_nasi.columns)}")


✅ Done! Rows: 4211, Columns: ['Date', 'Code', 'Name', '12m low', '12m high', 'Day low', 'Day high', 'Day price', 'Previous', 'Change', 'Change%', 'Volume', 'Adjust', 'Adjusted', 'Adjusted price', 'CBR_Rate', 'TBill_91D', 'Inflation_Rate']


In [6]:
final_nasi[['Date', 'CBR_Rate', 'TBill_91D', 'Inflation_Rate']].tail(10)

,Date,CBR_Rate,TBill_91D,Inflation_Rate
4201,2025-11-06,10.75,10.2,3.5
4202,2025-11-07,10.75,10.2,3.5
4203,2025-11-08,10.75,10.2,3.5
4204,2025-11-09,10.75,10.2,3.5
4205,2025-12-02,10.75,10.2,3.5
4206,2025-12-03,10.75,10.2,3.5
4207,2025-12-05,10.75,10.2,3.5
4208,2025-12-06,10.75,10.2,3.5
4209,2025-12-08,10.75,10.2,3.5
4210,2025-12-09,10.75,10.2,3.5


In [ ]:
# Consolidate adjusted price columns and drop unnecessary columns
nasi_data = final_nasi.copy()

# 1. Create a single Adjusted_Close column (consolidate the three adjusted price columns)
# Priority: Adjust > Adjusted > Adjusted price
nasi_data['Adjusted_Close'] = nasi_data['Adjust'].fillna(nasi_data['Adjusted']).fillna(nasi_data['Adjusted price'])

# Convert to numeric (removing any non-numeric values)
nasi_data['Adjusted_Close'] = pd.to_numeric(nasi_data['Adjusted_Close'], errors='coerce')

# If no adjusted close available, use Day price
nasi_data['Adjusted_Close'] = nasi_data['Adjusted_Close'].fillna(nasi_data['Day price'])

# 2. Keep only necessary columns for analysis
columns_to_keep = [
    'Date', 
    'Code', 
    'Day price',
    'Adjusted_Close',
    'Day low',
    'Day high',
    '12m low',
    '12m high',
    'CBR_Rate'
]

nasi_data = nasi_data[columns_to_keep]

# 3. Display info about cleaned data
print("✓ Data cleaned successfully!")
print(f"\nShape: {nasi_data.shape}")
print("\nColumns kept:")
print(nasi_data.columns.tolist())
print("\nFirst 5 rows:")
print(nasi_data.head())
print("\nData types:")
print(nasi_data.dtypes)
print(f"\nMissing values:\n{nasi_data.isnull().sum()}")

# Save cleaned data
output_path = r"C:\Users\thion\Kenya-Monetary-Policy-Stock-Market\notebooks\data\processed\nasi_cleaned.csv"
nasi_data.to_csv(output_path, index=False)
print(f"\n✓ Cleaned data saved to: {output_path}")